<table>
    <tr>
      <td>Miner?a de datos y paradigma BigData - Facultad de Inform?tica - UCM
      </td>
      <td>
      <img src="https://biblioteca.ucm.es/data/cont/media/www/pag-88746//escudo.jpg"  width=50/>
      </td>
     </tr>
</table>



## Pr?ctica 4 - Preprocesamiento de datos: an?lisis de tuits

Material original de Rafael Caballero, modificada y extendida por Pablo C. Ca?izares

---
## Introduccion

En esta practica trabajamos con el dataset **TUSA 2020** (tuits) para recorrer un flujo completo de adquisicion y preparacion de datos con `pandas`: carga desde un CSV comprimido, tipado correcto (fechas e identificadores), analisis visual y cuantitativo de valores faltantes, particion del conjunto en retweets vs tuits originales, limpieza guiada por criterio (columnas inutiles, constantes o especificas de un subconjunto) y un primer feature engineering (longitud del tuit) para detectar y filtrar outliers. El objetivo es aprender a leer el dataset con lupa: entender que significa cada campo, cuando aparece y como condiciona el preprocesado.

### Analisis del dataset
| Campo | Tipo recomendado | Que representa |
|---|---|---|
| `_id` | `str` | Identificador unico del tuit |
| `text` | `str` | Texto del tuit |
| `userid` | `str` | Identificador del usuario |
| `screen_name` | `str` | Nombre de usuario en Twitter |
| `created_at` | `datetime64[ns]` | Fecha/hora de publicacion |
| `lang` | `category` o `str` | Idioma detectado del tuit |
| `quote_count` | `int` | Numero de citas del tuit |
| `reply_count` | `int` | Numero de respuestas |
| `retweet_count` | `int` | Numero de retuits |
| `favorite_count` | `int` | Numero de "me gusta" |
| `coordinates` | `object` | Coordenadas geograficas |
| `reply` | `bool` | Indica si es respuesta |
| `quote` | `bool` | Indica si es cita |
| `RT` | `bool` | Indica si es retweet |
| `RT_source` | `str` | Id del tuit original (si es RT) |
| `nRTin` | `int` | Numero de retweets del tuit incluidos en el dataset |
| `textlen` | `int` | Longitud del texto del tuit |

---
Antes de empezar. Ejecutar el siguiente c?digo para instalar la biblioteca `missingno`. Tras hacerlo conviene reiniciar el entorno Jupyter

In [ ]:
modules = ["missingno","matplotlib"]


import sys
import os.path
from subprocess import check_call
import importlib
import os

def instala(modules):
    print("Instalando m?dulos")
    for m in modules:
        torch_loader = importlib.util.find_spec(m)
        if torch_loader is not None:
              print(m," encontrado")
        else:
            print(m," No encontrado, instalando...",end="")  
            try:        
                r = check_call([sys.executable, "-m", "pip", "install", "--user", m])
                print("?hecho!")
            except:
                print("?Problema al instalar ",m,"! ?seguro que el m?dulo existe?",sep="")

    print("?Terminado, quiz?s haya que hacer Restart!")

instala(modules)  

---
### **Ejercicio 1**. Leer en un dataframe `df` el fichero csv alojado en una web externa

"https://github.com/RafaelCaballero/tdm/raw/master/datos/tusa2020bcsv.zip"

Teniendo en cuenta que:
1.- Para cargarlo, debes utilizar `pd.read_csv` con la url proporcionada. Ojo, que el ejercicio no termina aqu?, tienes que tener en cuenta los dos siguientes puntos.

2.- La columna `created_at` debe ser le?da como una fecha. 

3.- Las columnas `_id`,`userid`, `RT_source` deben ser le?das como un string.


Notas: En [read_csv](https://pandas.pydata.org/docs/reference/api/pandas.read_csv.html):
* La opci?n `parse_dates`  permite leer fechas. Ej: `parse_dates=[column_name]`
* La opci?n `dtype`  nos permitir? indicar que `_id`,`userid`, `RT_source` son de tipo `str`. Ej: `dtype={column_name:str}`

In [ ]:
#soluci?n
import pandas as pd

url="https://github.com/RafaelCaballero/tdm/raw/master/datos/tusa2020bcsv.zip"

df = pd.read_csv(url,dtype={'_id' : str,'userid' : str,'RT_source' : str}, parse_dates=['created_at'])


# para probar
df.info()

In [ ]:
# para probar
df.head()

Metadatos: 

    _id: identificador del tweet, es ?nico
    text: texto del tweet
    userid: identificador del usuario, se puede repetir porque el mismo usuario puede tener varios tweets           
    screen_name: nombre en twitter del usuario
    created_at: fecha de emisi?n del tweet
    lang: idioma del tweet
    quote_count, reply_count, retweet_count, favorite_count: n?mero de citas, respuestas, retweets y marcas como favorito de este tweet, respectivamente
    coordinates: coordenadas desde las que se emiti?
    reply: True si el tweet es un reply, False en otro caso
    quote: True si el tweet es una cita, False en otro caso
    RT: True si el tweet es un retweet, False en otro caso
    RT_source: Si RT==True  indica el `_id` del tweet original             
    nRTin: n?mero de retweets del tweet inclu?dos en este conjunto

---
### **Ejercicio 2** 

a) A partir de `df.info()` seleccionar una columna candidata para ser borrada y borrarla. El resultado debe quedar en un dataframe `df2`
PD: No deb?is seleccionarla de manera program?tica, ?nicamente analizar el resultado de `df.info()` y utilizar ese cerebrazo que ten?is.

In [ ]:
#soluci?n
df2 = df.drop(columns = ['coordinates'])
# para probar
df2.info()

b) Utilizar la biblioteca `missingno` para mostrar gr?ficamente la matriz de nulos (m?todo matrix) en `df2`

In [ ]:
import matplotlib.pyplot as plt
import missingno as msno
df2.isna().sum()
# soluci?n
msno.matrix(df2)
plt.show()

?Extraes alguna conclusi?n  a partir de la matriz? (no hace falta que sea una certeza, solo escribir si algo nos llama la atenci?n). Una o 2 frases deben bastar

Comentario:



---
### **Ejercicio 3** 

Escribir una funci?n `columnas_con_nulos` con las siguientes caracter?sticas:

- Nombre: columnas_con_nulos
- Par?metros: df, un dataframe
- Salida: lista con los nombres de columnas de df que tienen alg?n nulo

Nota: si se usan bucles for la pregunta punt?a sobre 0.5



In [ ]:
# soluci?n
def columnas_con_nulos(df):
    lista = []
    serie=df.isnull().any()
    columnas_true = serie[serie].index.tolist()
    return columnas_true

# para probar
colnulos = columnas_con_nulos(df2)
colnulos

---

### **Ejercicio 4** 
De nuevo con la biblioteca `msno` mostrar un heatmap de las columnas de df2 que tienen alg?n nulo, representadas por la variable `colnulos` dada a continuaci?n

In [ ]:
colnulos = ['text', 'lang', 'quote_count', 'reply_count', 'retweet_count',
       'favorite_count', 'reply', 'quote', 'RT_source', 'nRTin']
msno.heatmap(df2[colnulos])
# soluci?n

?Qu? conclusi?n sacas? Una o dos frases deben bastar 

---

### **Ejercicio 5** 

Vamos a dividir `df2` en dos dataframes: `df_original` que solo tiene tweets que tengan `RT` a False, y `df_RT`que solo tenga filas que tengan `RT`a True.  Observar que es una partici?n del dataframe ya que RT no tiene nulos y solo toma los valores True y False

In [ ]:
# soluci?n

df_original=df2[df2['RT']==False]
df_RT=df2[df2['RT']==True]

In [ ]:
# para probar
print(len(df2), len(df_RT)+len(df_original))
df_original.info()

In [ ]:
# para probar
df_RT.info()

---
### **Ejercicio 6** 
Decidir a partir del resultado de las llamadas a info() (y del sentido com?n) qu? columnas se pueden eliminar en cada caso. Eliminarlas dejando el resultado en el mismo dataframe.

Pista: Las columnas que tienen un valor ?nico, tambien podr?an borrarse 
Nota: 
- No borrar "_id", "userid" o "screen_name", vamos a utlizarlas m?s adelante
- se pueden escribir los nombres de las columnas expl?citamente ("a mano"), pero entonces el apartado vale 0.25 en lugar de 1, es preferible programar el criterio que se haya elegido para su eliminaci?n


In [ ]:
df_RT.isna().all()
df_original.isna().all()

In [ ]:
# soluci?n
#df2 = df.drop(columns = ['coordinates'])
df_RT = df_RT.drop(columns = ['text', 'lang', 'quote_count', 'reply_count', 'retweet_count', 'favorite_count', 'reply', 'quote', 'nRTin'])
df_original = df_original.drop(columns = ['RT_source'])


In [ ]:
# para probar
df_RT.head()

In [ ]:
# para probar
df_original.head()

---

### **Ejercicio 7**. 
(3 puntos) Queremos analizar los perfiles de las personas que escribieron los tuits, por lo que vamos a analizar varios aspectos.

a) A?ade una nueva columna que represente la longitud de los tuits.

In [ ]:
#soluci?n
dflen= df.copy()
dflen['len_text'] = dflen['text'].str.len()
#para probar
dflen.head()

b) C?lculo de outliers de manera num?rica de la variable que hace referencia a la longitud de texto. 

In [ ]:
#soluci?n
# Calcular IQR y l?mites para outliers
IRQ=dflen['len_text'].quantile(0.75)-dflen['len_text'].quantile(0.25)
lower_bound=max(dflen['len_text'].quantile(0.25)-1.5*IRQ,0)
upper_bound=dflen['len_text'].quantile(0.75)+1.5*IRQ
print("Punto minimo: ", lower_bound)
print("Punto maximo: ", upper_bound)

# Filtrar outliers bajos y altos por separado
low_outliers = dflen[dflen['len_text'] < lower_bound]
high_outliers = dflen[dflen['len_text'] > upper_bound]

# Mostrar resultados
print("Outliers bajos:",low_outliers.shape[0])


print("\nOutliers altos: ",high_outliers.shape[0])
print(high_outliers['len_text'])



c) Dibuja un histograma para visualizar la distribucion de los textos

In [ ]:
import matplotlib.pyplot as plt

#soluci?n
plt.hist(dflen['len_text'])
plt.xlabel('Longitud del texto')
plt.ylabel('Frecuencia')
plt.title('Distribuci?n de la longitud de los tuits')
plt.show()

La distribuci?n es un poco extra?a ... ?Hay tuits vacios? ?Qu? ocurre?

In [ ]:
#Si, gran parte de los tuits tienen longitud nula y la mayoria no llega a penas a 200 aunque hay algunos que superan los 1600 caracteres

d) Dibuja boxplots para visualizar los posibles outliers 

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

#soluci?n
sns.boxplot(data=dflen, y='len_text')
plt.title('Boxplot de longitud de tuits')
plt.ylabel('Longitud del texto')
plt.show()


Ahora, muestra los outliers por idioma

In [ ]:
#soluci?n
sns.boxplot(data=dflen, x='lang', y='len_text')
plt.title('Distribuci?n de longitud de tuits por idioma')
plt.xlabel('Idioma')
plt.ylabel('Longitud del texto')
plt.show()

Vemos que algunos de los idiomas no son especialmente representativos. Quiz? deber?amos hacer algo con ello, pero antes ... 

e) Filtra los outliers

In [ ]:
#soluci?n
dflen_sin_outliers = dflen[(dflen['len_text'] >= lower_bound) & (dflen['len_text'] <= upper_bound)]
sns.boxplot(data=dflen_sin_outliers, x='lang', y='len_text')
plt.title('Distribuci?n de longitud de tuits por idioma (sin outliers)')
plt.xlabel('Idioma')
plt.ylabel('Longitud del texto')
plt.show()

No te alarmes, los outliers son efecto visual y reajuste de pesos, est? bien.

---
### **Ejercicio 8** 

Queremos eliminar de `df_original` las filas que corresponan a lenguajes que aparezcan menos de 10 veces en este dataframe. Dejar el resultado `df_original_lang`

Ayuda: 

- El m?todo df.c.value_counts() devuelve un valor de tipo Series donde el ?ndice (index) es el cada valor distinto de c, ordenado de m?s a menos frecuente, y el valor es dicha frecuencia
- El m?todo df.c.isin(l) devuelve True para aquellos valores de c que est?n en la lista l, y False en caso contrario

In [ ]:
# soluci?n
serie = df_original.lang.value_counts()
lista=serie[serie>=10].index.tolist()
a = df_original.lang.isin(lista)
lista2=a[a==False].index.tolist()
df_original_lang=df_original.drop(lista2)
# para probar
df_original_lang.lang.value_counts()